# SpanMarker NER — Evaluation Notebook

**Goal:** Evaluate the SpanMarker NER model on a PII-focused test set, **before** adding GLiNER to the pipeline.

This notebook is part of the **AI Privacy Gateway** project — Phase: Detection Layer Evaluation.

## How to use this notebook

1. Run all cells top-to-bottom.
2. By default, it runs on a **built-in 12-example test set** so you can verify the pipeline works.
3. To evaluate on your real dataset: drop your `eval_set.jsonl` file into the Kaggle dataset, then change `DATASET_PATH` in the Configuration cell.
4. The dataset format and the prompt to generate it are in `dataset_generation_prompt.txt` (downloaded alongside this notebook).
5. Metrics are saved to `spanmarker_metrics.csv` for later comparison with GLiNER.

## Model

`tomaarsen/span-marker-bert-base-fewnerd-fine-super` — SpanMarker with BERT-base, trained on FewNERD (66 fine-grained entity types). State-of-the-art span-based NER at this size (~440MB).

## What this notebook measures

- **Strict entity-level F1** — entity is correct only if start, end, AND label all match
- **Partial entity-level F1** — entity is correct if overlap > 0 AND label matches (forgives boundary errors)
- **Per-entity-type breakdown** — which entity types SpanMarker handles well, which it doesn't
- **Latency** — median + P95 inference time (CPU and GPU if available)


## 1. Install Dependencies

In [ ]:
# Install on Kaggle (these are not pre-installed)
# NOTE: Kaggle notebooks already have torch + transformers, so we only add spanmarker + seqeval
!pip install -q spanmarker seqeval 2>&1 | tail -5
print("✅ Dependencies installed")


## 2. Imports

In [ ]:
import json
import time
import statistics
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Any

import torch
from span_marker import SpanMarkerModel
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import Entity

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 3. Configuration

Change `DATASET_PATH` to point to your generated JSONL file.


In [ ]:
# ─── Configuration ───────────────────────────────────────────────
# Model: SpanMarker trained on FewNERD (66 fine-grained types)
MODEL_NAME = "tomaarsen/span-marker-bert-base-fewnerd-fine-super"

# Dataset path — set to None to use the built-in test set
# On Kaggle: put eval_set.jsonl in a Dataset, mount it, then set the path here.
# Example: DATASET_PATH = "/kaggle/input/my-pii-eval-set/eval_set.jsonl"
DATASET_PATH = None  # ← change this to your dataset path

# Output directory (Kaggle working directory is writable)
OUTPUT_DIR = Path("/kaggle/working/") if Path("/kaggle/working/").exists() else Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Confidence threshold for predictions (lower = more recall, less precision)
# SpanMarker predictions below this score are discarded
CONFIDENCE_THRESHOLD = 0.5

# Number of samples to display in the visual comparison
N_SAMPLES_TO_DISPLAY = 5

print(f"Model: {MODEL_NAME}")
print(f"Dataset: {'Built-in test set' if DATASET_PATH is None else DATASET_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")


## 4. Built-in Test Set (12 examples)

Used when `DATASET_PATH = None`. Covers PERSON, ORGANIZATION, LOCATION, DATE, EMAIL, PHONE, MONEY, and a few hard cases (ambiguous org names, multi-word person names).

Each row is: `{"text": "...", "entities": [[start_char, end_char, "LABEL"], ...]}`


In [ ]:
# ─── Built-in Test Set (12 examples) ───────────────────────────────
# Format: {"text": str, "entities": [[start_char, end_char_exclusive, "LABEL"], ...]}
# NOTE: end_char is EXCLUSIVE (Python slice convention).

BUILTIN_TEST_SET = [
    {
        "text": "Email John Smith at john@acme.com about the Q3 report.",
        "entities": [
            [6, 16, "PERSON"],
            [20, 33, "EMAIL"],
        ],
    },
    {
        "text": "Sarah Connor from Cyberdyne Systems called about the T-800 prototype.",
        "entities": [
            [0, 13, "PERSON"],
            [19, 37, "ORGANIZATION"],
        ],
    },
    {
        "text": "The meeting with Apple is scheduled for March 15, 2024 in Cupertino.",
        "entities": [
            [22, 27, "ORGANIZATION"],
            [48, 61, "DATE"],
            [65, 75, "LOCATION"],
        ],
    },
    {
        "text": "Please transfer $5,000 to IBAN DE89 3704 0044 0532 0130 00 by Friday.",
        "entities": [
            [21, 28, "MONEY"],
            [45, 51, "DATE"],
        ],
    },
    {
        "text": "Dr. Maria Rodriguez will present at the Stanford Medical Center on Tuesday.",
        "entities": [
            [4, 20, "PERSON"],
            [44, 69, "ORGANIZATION"],
            [73, 80, "DATE"],
        ],
    },
    {
        "text": "Contact support at +1-800-555-1234 or visit https://example.com for help.",
        "entities": [
            [20, 35, "PHONE_NUMBER"],
        ],
    },
    {
        "text": "Liu Wei from Beijing said the project deadline is October 31st.",
        "entities": [
            [0, 7, "PERSON"],
            [13, 20, "LOCATION"],
            [49, 60, "DATE"],
        ],
    },
    {
        "text": "The new iPhone 15 was announced at Apple Park in Cupertino, California.",
        "entities": [
            [8, 17, "PRODUCT"],
            [41, 51, "ORGANIZATION"],
            [55, 65, "LOCATION"],
            [67, 77, "LOCATION"],
        ],
    },
    {
        "text": "Albert Einstein published the theory of relativity in 1905 while working in Bern.",
        "entities": [
            [0, 15, "PERSON"],
            [67, 71, "DATE"],
            [89, 93, "LOCATION"],
        ],
    },
    {
        "text": "Call Jacques Dupont at +33 1 42 86 82 00 for the Paris office.",
        "entities": [
            [5, 19, "PERSON"],
            [23, 41, "PHONE_NUMBER"],
            [51, 56, "LOCATION"],
        ],
    },
    {
        "text": "Amazon Web Services announced a $1.2 billion investment in Dublin on Monday.",
        "entities": [
            [0, 19, "ORGANIZATION"],
            [44, 56, "MONEY"],
            [60, 67, "LOCATION"],
            [71, 77, "DATE"],
        ],
    },
    {
        "text": "Akira Tanaka from Sony will demonstrate the new PlayStation 5 at Tokyo Game Show.",
        "entities": [
            [0, 12, "PERSON"],
            [18, 22, "ORGANIZATION"],
            [58, 71, "PRODUCT"],
            [75, 90, "EVENT"],
        ],
    },
]

print(f"Built-in test set: {len(BUILTIN_TEST_SET)} examples")
print(f"Total entities: {sum(len(ex['entities']) for ex in BUILTIN_TEST_SET)}")


## 5. Load Dataset

Loads from JSONL if `DATASET_PATH` is set, otherwise uses the built-in test set.

**Expected JSONL format (one line per example):**
```json
{"text": "Email John at john@acme.com", "entities": [[6, 10, "PERSON"], [15, 29, "EMAIL"]]}
```

- `text`: full prompt text
- `entities`: list of `[start_char, end_char_exclusive, "LABEL"]` triples
- `LABEL`: uppercase entity type name (PERSON, ORGANIZATION, LOCATION, etc.)


In [ ]:
def load_dataset(path: str | None) -> List[Dict]:
    """Load JSONL dataset. Falls back to built-in test set if path is None."""
    if path is None:
        print("ℹ️  DATASET_PATH is None — using built-in test set")
        return BUILTIN_TEST_SET

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                ex = json.loads(line)
                # Validate schema
                assert "text" in ex and isinstance(ex["text"], str)
                assert "entities" in ex and isinstance(ex["entities"], list)
                for e in ex["entities"]:
                    assert len(e) == 3, f"Entity must be [start, end, label]: {e}"
                    assert isinstance(e[0], int) and isinstance(e[1], int)
                    assert isinstance(e[2], str)
                    # Validate offsets are within text bounds
                    assert 0 <= e[0] < e[1] <= len(ex["text"]), \
                        f"Span out of bounds in line {i}: {e} (text length={len(ex['text'])})"
                    # Validate the slice matches (sanity check)
                    actual = ex["text"][e[0]:e[1]]
                    # Note: we don't enforce that the slice equals anything specific —
                    # we just check the offsets are valid.
                examples.append(ex)
            except Exception as e:
                print(f"⚠️  Skipping line {i}: {e}")
                continue

    print(f"✅ Loaded {len(examples)} examples from {path}")
    print(f"   Total entities: {sum(len(ex['entities']) for ex in examples)}")
    return examples


# Load it
DATASET = load_dataset(DATASET_PATH)

# Quick stats
ent_types = defaultdict(int)
for ex in DATASET:
    for _, _, label in ex["entities"]:
        ent_types[label] += 1
print(f"\nEntity type distribution:")
for label, count in sorted(ent_types.items(), key=lambda x: -x[1]):
    print(f"  {label:25s} {count:4d}")


## 6. Load SpanMarker Model

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("⏳ This may take 30-60 seconds on first run (downloads ~440 MB)...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

t0 = time.time()
model = SpanMarkerModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"✅ Model loaded in {time.time() - t0:.1f}s")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


## 7. Run Inference

Run SpanMarker on every example. Each prediction returns:
- `start`, `end`: character span offsets
- `label`: predicted entity type
- `score`: confidence (0-1)


In [ ]:
def run_inference(model, examples: List[Dict], threshold: float = 0.5) -> Tuple[List[Dict], List[float]]:
    """Run SpanMarker on all examples. Returns (predictions, latencies_ms)."""
    all_preds = []
    latencies = []

    for i, ex in enumerate(examples):
        text = ex["text"]

        t0 = time.time()
        with torch.no_grad():
            raw_preds = model.predict(text)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        # Normalize SpanMarker output to [start, end, label, score]
        preds = []
        for p in raw_preds:
            # SpanMarker returns dict with "span": [start, end], "label", "score"
            start, end = p["span"]
            label = p["label"]
            score = p.get("score", 1.0)
            if score >= threshold:
                preds.append({
                    "start": int(start),
                    "end": int(end),
                    "label": label,
                    "score": float(score),
                    "text": text[start:end],
                })
        all_preds.append(preds)

        if (i + 1) % 25 == 0 or i == len(examples) - 1:
            print(f"  [{i+1}/{len(examples)}]  last={latency_ms:.1f}ms")

    return all_preds, latencies


print(f"Running inference on {len(DATASET)} examples...")
t0 = time.time()
PREDICTIONS, LATENCIES = run_inference(model, DATASET, CONFIDENCE_THRESHOLD)
total_s = time.time() - t0
print(f"\n✅ Done in {total_s:.1f}s ({total_s/len(DATASET)*1000:.1f}ms/example avg)")


## 8. Latency Benchmark

In [ ]:
lat_sorted = sorted(LATENCIES)
n = len(lat_sorted)
median = statistics.median(lat_sorted)
p95 = lat_sorted[int(n * 0.95)] if n >= 20 else max(lat_sorted)
p99 = lat_sorted[int(n * 0.99)] if n >= 100 else max(lat_sorted)

print(f"Latency statistics ({n} examples, device={device}):")
print(f"  Median:  {median:7.1f} ms")
print(f"  P95:     {p95:7.1f} ms")
print(f"  P99:     {p99:7.1f} ms")
print(f"  Min:     {min(lat_sorted):7.1f} ms")
print(f"  Max:     {max(lat_sorted):7.1f} ms")
print(f"  Mean:    {sum(lat_sorted)/n:7.1f} ms")
print()
print(f"vs. project budget: P95 ≤ 400ms (combined with GLiNER later)")
print(f"  → SpanMarker alone uses {p95/400*100:.0f}% of the P95 budget")


## 9. Entity-Level Metrics

Two scoring modes:

- **Strict** — entity is "correct" only if `(start, end, label)` all match exactly
- **Partial** — entity is "correct" if spans overlap AND labels match (forgives boundary errors)

Both use the standard NER definition: TP / (TP + FP) = precision, TP / (TP + FN) = recall.


In [ ]:
def spans_overlap(a: Tuple[int, int], b: Tuple[int, int]) -> bool:
    """Check if two (start, end) spans overlap."""
    return a[0] < b[1] and b[0] < a[1]


def compute_metrics(examples, predictions, mode="strict"):
    """
    Compute entity-level precision, recall, F1.
    mode: 'strict' (exact match) or 'partial' (any overlap + label match)
    """
    tp = fp = fn = 0

    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]

        matched_gold = set()
        matched_pred = set()

        # For each prediction, find a matching gold (greedy)
        for i, p in enumerate(pred):
            for j, g in enumerate(gold):
                if j in matched_gold:
                    continue
                if p[2] != g[2]:  # labels must match
                    continue
                if mode == "strict":
                    if p[0] == g[0] and p[1] == g[1]:
                        tp += 1
                        matched_gold.add(j)
                        matched_pred.add(i)
                        break
                else:  # partial
                    if spans_overlap((p[0], p[1]), (g[0], g[1])):
                        tp += 1
                        matched_gold.add(j)
                        matched_pred.add(i)
                        break
            else:
                # No match found for this prediction → false positive
                fp += 1

        # Unmatched golds → false negatives
        fn += len(gold) - len(matched_gold)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "mode": mode,
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# Compute both modes
metrics_strict = compute_metrics(DATASET, PREDICTIONS, mode="strict")
metrics_partial = compute_metrics(DATASET, PREDICTIONS, mode="partial")

print("╔══════════════════════════════════════════════════════════╗")
print("║              ENTITY-LEVEL METRICS                       ║")
print("╠══════════════════════╦════════════╦════════════╦═══════╣")
print("║ Metric               ║  Strict    ║  Partial   ║       ║")
print("╠══════════════════════╬════════════╬════════════╬═══════╣")
print(f"║ Precision            ║  {metrics_strict['precision']:.4f}   ║  {metrics_partial['precision']:.4f}    ║       ║")
print(f"║ Recall               ║  {metrics_strict['recall']:.4f}   ║  {metrics_partial['recall']:.4f}    ║       ║")
print(f"║ F1                   ║  {metrics_strict['f1']:.4f}   ║  {metrics_partial['f1']:.4f}    ║       ║")
print(f"║ True Positives       ║  {metrics_strict['tp']:>5}     ║  {metrics_partial['tp']:>5}      ║       ║")
print(f"║ False Positives      ║  {metrics_strict['fp']:>5}     ║  {metrics_partial['fp']:>5}      ║       ║")
print(f"║ False Negatives      ║  {metrics_strict['fn']:>5}     ║  {metrics_partial['fn']:>5}      ║       ║")
print("╚══════════════════════╩════════════╩════════════╩═══════╝")


## 10. Per-Entity-Type Breakdown

In [ ]:
def compute_per_type_metrics(examples, predictions, mode="strict"):
    """Compute precision/recall/F1 for each entity type separately."""
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]

        matched_gold = set()
        for p_start, p_end, p_label in pred:
            for j, (g_start, g_end, g_label) in enumerate(gold):
                if j in matched_gold or g_label != p_label:
                    continue
                if mode == "strict":
                    if p_start == g_start and p_end == g_end:
                        per_type[p_label]["tp"] += 1
                        matched_gold.add(j)
                        break
                else:
                    if spans_overlap((p_start, p_end), (g_start, g_end)):
                        per_type[p_label]["tp"] += 1
                        matched_gold.add(j)
                        break
            else:
                per_type[p_label]["fp"] += 1

        for j, (_, _, g_label) in enumerate(gold):
            if j not in matched_gold:
                per_type[g_label]["fn"] += 1

    # Compute F1 per type
    rows = []
    for label, counts in per_type.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        rows.append({
            "entity_type": label,
            "tp": tp, "fp": fp, "fn": fn,
            "support": tp + fn,  # total gold instances
            "precision": p, "recall": r, "f1": f1,
        })
    rows.sort(key=lambda r: -r["support"])
    return rows


# Compute per-type (strict mode — that's the standard NER metric)
per_type_strict = compute_per_type_metrics(DATASET, PREDICTIONS, mode="strict")

print(f"\nPer-Entity-Type (STRICT mode):")
print(f"{'Entity Type':<25} {'Support':>8} {'TP':>5} {'FP':>5} {'FN':>5} {'P':>7} {'R':>7} {'F1':>7}")
print("─" * 78)
for r in per_type_strict:
    print(f"{r['entity_type']:<25} {r['support']:>8} {r['tp']:>5} {r['fp']:>5} {r['fn']:>5} "
          f"{r['precision']:>7.3f} {r['recall']:>7.3f} {r['f1']:>7.3f}")
print("─" * 78)
total_support = sum(r["support"] for r in per_type_strict)
total_tp = sum(r["tp"] for r in per_type_strict)
total_fp = sum(r["fp"] for r in per_type_strict)
total_fn = sum(r["fn"] for r in per_type_strict)
print(f"{'TOTAL':<25} {total_support:>8} {total_tp:>5} {total_fp:>5} {total_fn:>5} "
      f"{total_tp/(total_tp+total_fp):>7.3f} {total_tp/(total_tp+total_fn):>7.3f} "
      f"{2*total_tp/(2*total_tp+total_fp+total_fn):>7.3f}")


## 11. Sample Predictions vs Ground Truth

Visualizes the first N examples. Green = correctly detected (TP), Red = missed (FN), Yellow = false positive.


In [ ]:
from IPython.display import HTML, display

COLORS = {
    "tp": "#10B981",   # green
    "fn": "#EF4444",   # red
    "fp": "#F59E0B",   # amber
    "gold_only": "#3B82F6",  # blue (ground truth baseline)
}


def render_example_html(text, gold, pred):
    """Render an example with colored highlights."""
    # Classify each gold and pred span
    gold_set = set((g[0], g[1], g[2]) for g in gold)
    pred_set = set((p["start"], p["end"], p["label"]) for p in pred)

    # Build a list of (start, end, type, label) sorted by start
    spans_to_render = []

    for g in gold:
        in_pred = g in pred_set
        # Also check partial overlap
        if not in_pred:
            for p in pred:
                if p["label"] == g[2] and spans_overlap((p["start"], p["end"]), (g[0], g[1])):
                    in_pred = True
                    break
        spans_to_render.append((g[0], g[1], "tp" if in_pred else "fn", g[2]))

    for p in pred:
        in_gold = (p["start"], p["end"], p["label"]) in gold_set
        if not in_gold:
            # Check partial
            for g in gold:
                if g[2] == p["label"] and spans_overlap((g[0], g[1]), (p["start"], p["end"])):
                    in_gold = True
                    break
        if not in_gold:
            spans_to_render.append((p["start"], p["end"], "fp", p["label"]))

    # Sort by start, then end
    spans_to_render.sort(key=lambda s: (s[0], s[1]))

    # Build HTML with overlapping-safe rendering (just sequential, non-overlapping for simplicity)
    html_parts = []
    last_end = 0
    for start, end, kind, label in spans_to_render:
        if start < last_end:
            start = last_end  # avoid overlap issues
        if start > last_end:
            html_parts.append(f"<span>{text[last_end:start]}</span>")
        color = COLORS[kind]
        html_parts.append(
            f'<span style="background-color: {color}33; '
            f'border-bottom: 2px solid {color}; padding: 1px 2px;" '
            f'title="{kind.upper()} - {label}">'
            f'{text[start:end]}'
            f'<sub style="font-size:9px;color:{color}"> [{label}]</sub>'
            f'</span>'
        )
        last_end = end
    if last_end < len(text):
        html_parts.append(f"<span>{text[last_end:]}</span>")

    return "".join(html_parts)


# Show N samples
print(f"\nShowing {N_SAMPLES_TO_DISPLAY} sample predictions:\n")
for i, (ex, preds) in enumerate(zip(DATASET[:N_SAMPLES_TO_DISPLAY], PREDICTIONS[:N_SAMPLES_TO_DISPLAY])):
    gold = ex["entities"]
    html = render_example_html(ex["text"], gold, preds)
    print(f"─── Example {i+1} ───")
    print(f"Text: {ex['text']}")
    print(f"Gold entities: {gold}")
    print(f"Predictions:  {[(p['start'], p['end'], p['label'], round(p['score'],2)) for p in preds]}")
    print()
    display(HTML(f'<div style="font-family: monospace; padding: 8px; background: #F8FAFC; border-radius: 4px;">{html}</div>'))
    print()

# Legend
print("\nLegend:")
display(HTML('''
<table style="font-family: monospace;">
<tr><td style="background:#10B98133;border-bottom:2px solid #10B981;padding:2px 8px">TP</td>
    <td>True Positive — correctly detected</td></tr>
<tr><td style="background:#EF444433;border-bottom:2px solid #EF4444;padding:2px 8px">FN</td>
    <td>False Negative — model missed this gold entity</td></tr>
<tr><td style="background:#F59E0B33;border-bottom:2px solid #F59E0B;padding:2px 8px">FP</td>
    <td>False Positive — model invented this entity</td></tr>
</table>
'''))


## 12. Error Analysis

Show all false positives and false negatives, so you can see exactly where SpanMarker fails.


In [ ]:
print("=" * 80)
print("FALSE NEGATIVES (gold entities SpanMarker missed)")
print("=" * 80)
fn_count = 0
for i, (ex, preds) in enumerate(zip(DATASET, PREDICTIONS)):
    gold = ex["entities"]
    pred_set = set((p["start"], p["end"], p["label"]) for p in preds)
    for g_start, g_end, g_label in gold:
        # Check exact match OR partial overlap
        matched = (g_start, g_end, g_label) in pred_set
        if not matched:
            for p in preds:
                if p["label"] == g_label and spans_overlap((p["start"], p["end"]), (g_start, g_end)):
                    matched = True
                    break
        if not matched:
            fn_count += 1
            print(f"  [{i+1}] GOLD: "{ex['text'][g_start:g_end]}" ({g_label})")
            print(f"        Context: ...{ex['text'][max(0,g_start-20):g_end+20]}...")
            print()
print(f"Total FN: {fn_count}")

print()
print("=" * 80)
print("FALSE POSITIVES (entities SpanMarker invented)")
print("=" * 80)
fp_count = 0
for i, (ex, preds) in enumerate(zip(DATASET, PREDICTIONS)):
    gold = ex["entities"]
    for p in preds:
        # Check exact match OR partial overlap
        matched = any(
            (p["start"], p["end"], p["label"]) == (g[0], g[1], g[2])
            for g in gold
        )
        if not matched:
            for g in gold:
                if g[2] == p["label"] and spans_overlap((g[0], g[1]), (p["start"], p["end"])):
                    matched = True
                    break
        if not matched:
            fp_count += 1
            print(f"  [{i+1}] PRED: "{p['text']}" ({p['label']}, score={p['score']:.2f})")
            print(f"        Context: ...{ex['text'][max(0,p['start']-20):p['end']+20]}...")
            print()
print(f"Total FP: {fp_count}")


## 13. Save Metrics

Save results to CSV files in the output directory. These can be downloaded from Kaggle and used as a baseline when you add GLiNER next.


In [ ]:
import csv

# ─── Save overall metrics ───
overall_path = OUTPUT_DIR / "spanmarker_overall_metrics.csv"
with open(overall_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "strict", "partial"])
    w.writerow(["precision", metrics_strict["precision"], metrics_partial["precision"]])
    w.writerow(["recall", metrics_strict["recall"], metrics_partial["recall"]])
    w.writerow(["f1", metrics_strict["f1"], metrics_partial["f1"]])
    w.writerow(["tp", metrics_strict["tp"], metrics_partial["tp"]])
    w.writerow(["fp", metrics_strict["fp"], metrics_partial["fp"]])
    w.writerow(["fn", metrics_strict["fn"], metrics_partial["fn"]])
    w.writerow(["n_examples", len(DATASET), len(DATASET)])
    w.writerow(["median_latency_ms", median, median])
    w.writerow(["p95_latency_ms", p95, p95])
    w.writerow(["device", device, device])
    w.writerow(["model", MODEL_NAME, MODEL_NAME])
    w.writerow(["confidence_threshold", CONFIDENCE_THRESHOLD, CONFIDENCE_THRESHOLD])
print(f"✅ Saved overall metrics: {overall_path}")

# ─── Save per-type metrics ───
per_type_path = OUTPUT_DIR / "spanmarker_per_type_metrics.csv"
with open(per_type_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["entity_type", "support", "tp", "fp", "fn", "precision", "recall", "f1"])
    for r in per_type_strict:
        w.writerow([r["entity_type"], r["support"], r["tp"], r["fp"], r["fn"],
                    r["precision"], r["recall"], r["f1"]])
print(f"✅ Saved per-type metrics: {per_type_path}")

# ─── Save raw predictions ───
raw_path = OUTPUT_DIR / "spanmarker_raw_predictions.jsonl"
with open(raw_path, "w") as f:
    for ex, preds in zip(DATASET, PREDICTIONS):
        out = {
            "text": ex["text"],
            "gold": ex["entities"],
            "pred": [[p["start"], p["end"], p["label"], round(p["score"], 4)] for p in preds],
        }
        f.write(json.dumps(out) + "\n")
print(f"✅ Saved raw predictions: {raw_path}")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Model:          {MODEL_NAME}")
print(f"Examples:       {len(DATASET)}")
print(f"Device:         {device}")
print(f"Strict F1:      {metrics_strict['f1']:.4f}")
print(f"Partial F1:     {metrics_partial['f1']:.4f}")
print(f"Strict P/R:     {metrics_strict['precision']:.4f} / {metrics_strict['recall']:.4f}")
print(f"Median latency: {median:.1f} ms")
print(f"P95 latency:    {p95:.1f} ms")
print(f"\nFiles saved to: {OUTPUT_DIR}")
print(f"\n👉 Next: share these metrics with the team and compare with GLiNER2-PII")
print(f"   when you wire it into the pipeline.")


## 14. Next Steps

### Now that you have SpanMarker baseline metrics:

1. **Generate a real evaluation dataset** using the prompt in `dataset_generation_prompt.txt` (downloaded alongside this notebook). Aim for 50-100 diverse examples.

2. **Re-run this notebook** with `DATASET_PATH` pointing to your real dataset.

3. **Compare with GLiNER2-PII** — duplicate this notebook, swap the model to `urchade/gliner2-PII`, run on the same dataset, compare CSVs side-by-side.

4. **Decide pipeline architecture** based on which model wins on which entity types:
   - If SpanMarker wins on PERSON/ORG → use SpanMarker first
   - If GLiNER wins on PII-specific types (EMAIL, PHONE, PASSPORT) → use GLiNER second
   - Run both, merge spans with dedup layer

5. **Latency check** — if SpanMarker alone is already >200ms P95, consider switching to a smaller SpanMarker variant (e.g., `span-marker-bert-base-cased` instead of FewNERD-large).

### Files produced by this notebook

- `spanmarker_overall_metrics.csv` — top-line numbers (P/R/F1, latency, device)
- `spanmarker_per_type_metrics.csv` — per-entity-type breakdown
- `spanmarker_raw_predictions.jsonl` — every example with gold + pred spans, for debugging

Download these from Kaggle (`Output` tab) after the run finishes.
